## 0. BenchMark_renewal 데이터 확인

In [47]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/MS_AI_NLP(2026)/05_모델학습및평가/BenchMark_renewal'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# train json 확인
train_df = pd.read_json(f'{DATA_DIR}/train.jsonl', lines=True)
print(train_df.shape)
train_df.head()

(9234, 10)


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,row_id
0,A1K582XYLTKUCD,B0042F1L4S,"Coder10 ""Ricardo""","[0, 1]",im not a profesional player but i like to play...,5,Everything a hobbiest want,1361404800,"02 21, 2013",8492
1,A2J4UAF6RW13WK,B000EELB8W,Michael W DeSilva,"[0, 0]",This is just an excellent product. I have been...,5,Excellent Product,1371686400,"06 20, 2013",4666
2,A3OXHLG6DIBRW8,B000BU5V58,"C. Hill ""CFH""","[1, 1]","We opted for the World Tour ""Guitar Gig Bag"" o...",4,Roomy Guitar Case - Recommended,1290556800,"11 24, 2010",4286
3,A3872Y2XH0YDX1,B000CZ0RLK,Amazon Customer,"[0, 0]","This is not a top tier condenser mic, but it i...",5,definitely a good value,1311552000,"07 25, 2011",4446
4,A2G3VQU2GRN8BU,B000978D58,"Moral Hazard ""D""","[1, 3]",This could be used for lightweight microphones...,3,Too cheap.,1316044800,"09 15, 2011",3921


In [49]:
# test json 확인
test_df = pd.read_json(f'{DATA_DIR}/test.jsonl', lines=True)
print(test_df.shape)
test_df.head()

(1027, 8)


,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,reviewTime,row_id
0,A30KA8I5AHSKJZ,B002024UDE,Steinbeckian,Sounds decent and stays in tune. For the pric...,For the money it is worth it,1358121600,"01 14, 2013",7056
1,A2206923NH9ZDI,B0016MJ1T2,Texas Buyer,The Seller shipped very fast and I give them 5...,Inexpensive alternative to harmony effects,1342051200,"07 12, 2012",6141
2,A1BNFL98NKJNYA,B000X31R2E,Leseagle,"Great Strap Button very solid, aluminum and is...",Great Strap Button... Solid,1400544000,"05 20, 2014",5873
3,A3SQ2WX9CAJCKZ,B000J5XS3C,"C. Erikson ""charlinus""",I just got this to run my passive pa. Its inex...,Easy to use,1326585600,"01 15, 2012",4806
4,A3W2KQ6CLTQ0ZD,B0002D0CLC,Ipodcrazy,I used to use Fender medium picks only. Now fo...,A classic,1355961600,"12 20, 2012",919


In [50]:
# submission csv 확인
submission_df = pd.read_csv(f'{DATA_DIR}/submission.csv')
print(submission_df.shape)
submission_df.head()

(1027, 4)


,row_id,helpful_yes_pred,helpful_total_pred,overall_pred
0,7056,0.163290,0.261697,4.493876
1,6141,0.244523,0.279981,4.363108
2,5873,2.058375,2.350251,4.514071
3,4806,1.435810,1.610781,4.633665
4,919,2.734200,3.077200,3.845367


## 1. 모델 학습 (train.jsonl 이용)

In [ ]:
import ast
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [52]:
# helpful 컬럼 helpful_yes, helpful_total로 분리
def parse_helpful(x):
    if isinstance(x, list) and len(x) == 2:
        return x[0], x[1]
    if isinstance(x, str):
        y = ast.literal_eval(x)
        return y[0], y[1]
    return 0, 0

train_df['helpful_yes'], train_df['helpful_total'] = zip(*train_df['helpful'].apply(parse_helpful))

# helpful_yes/helpful_total은 값의 범위가 넓고(0~수십 표) 분포가 한쪽으로 치우쳐 있어,
# 그대로 MSE에 넣으면 overall(1~5점)보다 loss/gradient를 지배해 소수의 인기 리뷰에 과적합되기 쉬움.
# log1p로 압축한 값을 학습 타깃으로 쓰고, 예측 후에는 expm1로 원래 스케일로 되돌린다.
train_df['helpful_yes_log'] = np.log1p(train_df['helpful_yes'])
train_df['helpful_total_log'] = np.log1p(train_df['helpful_total'])

train_df[['helpful', 'helpful_yes', 'helpful_total', 'helpful_yes_log', 'helpful_total_log']].head()

,helpful,helpful_yes,helpful_total,helpful_yes_log,helpful_total_log
0,"[0, 1]",0,1,0.000000,0.693147
1,"[0, 0]",0,0,0.000000,0.000000
2,"[1, 1]",1,1,0.693147,0.693147
3,"[0, 0]",0,0,0.000000,0.000000
4,"[1, 3]",1,3,0.693147,1.386294


In [ ]:
# BERT 토크나이저로 reviewText와 summary를 토큰화하고, 최대 길이로 패딩 (input_ids + attention_mask)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN_REVIEW = 100
MAX_LEN_SUMMARY = 10

def preprocessing(text, max_length):
    if pd.isna(text):
        text = ''
    result = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='np'
    )
    return result['input_ids'][0], result['attention_mask'][0]

train_df['reviewText_token'], train_df['reviewText_mask'] = zip(*train_df['reviewText'].apply(lambda x: preprocessing(x, MAX_LEN_REVIEW)))
train_df['summary_token'], train_df['summary_mask'] = zip(*train_df['summary'].apply(lambda x: preprocessing(x, MAX_LEN_SUMMARY)))
train_df[['reviewText_token', 'reviewText_mask', 'summary_token', 'summary_mask']].head()

In [54]:
# Feature Engineering: 텍스트 길이/문장부호/대문자 비율 등 보조 수치 feature와 작성 연도 feature 추가
def extract_text_features(text):
    if pd.isna(text):
        text = ''
    words = text.split()
    n_words = len(words)
    n_chars = len(text)
    n_exclaim = text.count('!')
    n_question = text.count('?')
    n_upper_words = sum(1 for w in words if len(w) > 1 and w.isupper())
    avg_word_len = n_chars / n_words if n_words > 0 else 0.0
    return n_words, n_chars, n_exclaim, n_question, n_upper_words, avg_word_len

REVIEW_FEAT_COLS = ['review_n_words', 'review_n_chars', 'review_n_exclaim', 'review_n_question', 'review_n_upper_words', 'review_avg_word_len']
SUMMARY_FEAT_COLS = ['summary_n_words', 'summary_n_chars', 'summary_n_exclaim', 'summary_n_question', 'summary_n_upper_words', 'summary_avg_word_len']
EXTRA_FEAT_COLS = REVIEW_FEAT_COLS + SUMMARY_FEAT_COLS + ['review_year']

def add_text_features(df):
    df[REVIEW_FEAT_COLS] = df['reviewText'].apply(lambda x: pd.Series(extract_text_features(x)))
    df[SUMMARY_FEAT_COLS] = df['summary'].apply(lambda x: pd.Series(extract_text_features(x)))
    df['review_year'] = pd.to_datetime(df['unixReviewTime'], unit='s').dt.year.astype(float)
    return df

train_df = add_text_features(train_df)
train_df[EXTRA_FEAT_COLS].describe()

,review_n_words,review_n_chars,review_n_exclaim,review_n_question,review_n_upper_words,review_avg_word_len,summary_n_words,summary_n_chars,summary_n_exclaim,summary_n_question,summary_n_upper_words,summary_avg_word_len,review_year
count,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000,9234.000000
mean,90.338965,486.723522,0.413580,0.093026,1.018627,5.344825,4.376760,24.388131,0.204787,0.012779,0.112194,5.744501,2012.662551
std,109.347447,601.801952,1.169374,0.434748,3.542883,0.429922,2.857689,15.669246,0.742466,0.133475,0.541944,1.525474,1.265840
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,2004.000000
25%,31.000000,164.000000,0.000000,0.000000,0.000000,5.087719,2.000000,13.000000,0.000000,0.000000,0.000000,4.857143,2012.000000
50%,54.000000,286.000000,0.000000,0.000000,0.000000,5.325000,4.000000,21.000000,0.000000,0.000000,0.000000,5.500000,2013.000000
75%,105.000000,554.000000,0.000000,0.000000,1.000000,5.575221,6.000000,32.000000,0.000000,0.000000,0.000000,6.375000,2014.000000
max,2043.000000,11310.000000,31.000000,8.000000,88.000000,10.800000,25.000000,128.000000,22.000000,3.000000,11.000000,27.000000,2014.000000


In [55]:
# train/validation 분리
train_split_df, val_split_df = train_test_split(train_df, test_size=0.2, random_state=42)
train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

len(train_split_df), len(val_split_df)

(7387, 1847)

In [ ]:
# 수치형 보조 feature 표준화 (평균 0, 분산 1) - StandardScaler를 train_split에만 fit해 val/test 누수 방지
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_split_df[EXTRA_FEAT_COLS])

def normalize_features(df):
    return scaler.transform(df[EXTRA_FEAT_COLS]).astype('float32')

train_split_df['extra_features'] = list(normalize_features(train_split_df))
val_split_df['extra_features'] = list(normalize_features(val_split_df))

In [ ]:
# 데이터셋 구성
class ReviewDataset(Dataset):
    def __init__(self, df):
        self.data = df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {
            'review_ids': torch.tensor(self.data['reviewText_token'][idx], dtype=torch.long),
            'review_mask': torch.tensor(self.data['reviewText_mask'][idx], dtype=torch.long),
            'summary_ids': torch.tensor(self.data['summary_token'][idx], dtype=torch.long),
            'summary_mask': torch.tensor(self.data['summary_mask'][idx], dtype=torch.long),
            'extra_features': torch.tensor(self.data['extra_features'][idx], dtype=torch.float),
            'overall': torch.tensor(self.data['overall'][idx], dtype=torch.float),
            'helpful': torch.tensor(self.data[['helpful_yes_log', 'helpful_total_log']].iloc[idx].values, dtype=torch.float)
        }

train_dataset = ReviewDataset(train_split_df)
val_dataset = ReviewDataset(val_split_df)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
# 모델 정의
# 직접 학습하는 LSTM 대신, 사전학습된 BERT(bert-base-uncased)를 리뷰/요약 공용 인코더로 사용.
# 7387건의 적은 데이터로 Transformer를 처음부터 학습하면 오히려 과적합이 심해지므로,
# 사전학습된 가중치 대부분은 freeze하고 마지막 n_unfrozen_layers개 encoder layer + pooler만 fine-tuning한다.
#
# - BERT의 768차원 출력을 그대로 concat하면 head 입력 차원이 커져 과적합되기 쉬우므로,
#   작은 FC(text_proj)로 한 번 투영해 hidden_dim을 줄인다 (review/summary 공용).
# - 수치형 보조 feature(extra_features)도 곧바로 concat하지 않고 FC + BatchNorm을 거쳐 변환하고,
#   학습 시에만 작은 가우시안 노이즈를 더해 숫자를 그대로 외우지 못하도록 한다.
# - 텍스트(BERT 내부 dropout)뿐 아니라 수치 branch에도 별도 dropout을 적용하고,
#   concat 직후에는 더 강한 dropout(0.4)을 적용한다. 최종 head는 깊게 쌓지 않고 Linear 한 층만 사용.

class TransformerRatingModel(nn.Module):
    def __init__(self, extra_feat_dim, pretrained_name='bert-base-uncased', n_unfrozen_layers=2,
                 text_proj_dim=128, numeric_dim=16, dropout=0.4, numeric_dropout=0.2, noise_std=0.01):
        super().__init__()
        self.bert = BertModel.from_pretrained(pretrained_name)
        bert_hidden_dim = self.bert.config.hidden_size

        for param in self.bert.parameters():
            param.requires_grad = False
        for layer in self.bert.encoder.layer[-n_unfrozen_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
        for param in self.bert.pooler.parameters():
            param.requires_grad = True

        self.text_proj = nn.Sequential(nn.Linear(bert_hidden_dim, text_proj_dim), nn.ReLU())

        self.numeric_fc = nn.Sequential(
            nn.Linear(extra_feat_dim, numeric_dim),
            nn.BatchNorm1d(numeric_dim),
            nn.ReLU(),
        )
        self.numeric_dropout = nn.Dropout(numeric_dropout)
        self.noise_std = noise_std

        self.dropout = nn.Dropout(dropout)
        combined_dim = text_proj_dim * 2 + numeric_dim
        self.overall = nn.Linear(combined_dim, 1)
        self.helpful = nn.Linear(combined_dim, 2)

    def encode_text(self, input_ids, attention_mask):
        pooled = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        return self.text_proj(pooled)

    def encode_numeric(self, extra_features):
        if self.training and self.noise_std > 0:
            extra_features = extra_features + torch.randn_like(extra_features) * self.noise_std
        numeric_repr = self.numeric_fc(extra_features)
        return self.numeric_dropout(numeric_repr)

    def forward(self, review_ids, review_mask, summary_ids, summary_mask, extra_features):
        review_repr = self.encode_text(review_ids, review_mask)
        summary_repr = self.encode_text(summary_ids, summary_mask)
        numeric_repr = self.encode_numeric(extra_features)

        combined_hidden = torch.cat([review_repr, summary_repr, numeric_repr], dim=1)
        combined_hidden = self.dropout(combined_hidden)

        overall_pred = self.overall(combined_hidden)
        helpful_pred = self.helpful(combined_hidden)

        return overall_pred, helpful_pred

In [ ]:
EXTRA_FEAT_DIM = len(EXTRA_FEAT_COLS)
N_UNFROZEN_LAYERS = 2
TEXT_PROJ_DIM = 128
NUMERIC_DIM = 16

model = TransformerRatingModel(
    EXTRA_FEAT_DIM,
    n_unfrozen_layers=N_UNFROZEN_LAYERS,
    text_proj_dim=TEXT_PROJ_DIM,
    numeric_dim=NUMERIC_DIM,
).to(device)
criterion = nn.MSELoss()

# fine-tuning하는 BERT 레이어는 작은 lr, 새로 추가한 투영/수치/head 레이어는 더 큰 lr로 분리 (discriminative learning rate)
bert_params = [p for p in model.bert.parameters() if p.requires_grad]
head_params = (
    list(model.text_proj.parameters())
    + list(model.numeric_fc.parameters())
    + list(model.overall.parameters())
    + list(model.helpful.parameters())
)
optimizer = optim.AdamW([
    {'params': bert_params, 'lr': 2e-5},
    {'params': head_params, 'lr': 1e-3},
], weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

### 학습 / 검증 루프

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            review_ids = batch['review_ids'].to(device)
            review_mask = batch['review_mask'].to(device)
            summary_ids = batch['summary_ids'].to(device)
            summary_mask = batch['summary_mask'].to(device)
            extra = batch['extra_features'].to(device)
            overall = batch['overall'].to(device)
            helpful = batch['helpful'].to(device)

            overall_pred, helpful_pred = model(review_ids, review_mask, summary_ids, summary_mask, extra)
            loss_overall = criterion(overall_pred, overall.view(-1, 1))
            loss_helpful = criterion(helpful_pred, helpful)
            loss = loss_overall + loss_helpful
            running_loss += loss.item()
    return running_loss / len(loader)


def train(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=20, patience=4, max_grad_norm=1.0):
    # val loss 기준으로 가장 좋았던 시점의 가중치를 저장해두고, patience 동안 개선이 없으면 조기 종료.
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            review_ids = batch['review_ids'].to(device)
            review_mask = batch['review_mask'].to(device)
            summary_ids = batch['summary_ids'].to(device)
            summary_mask = batch['summary_mask'].to(device)
            extra = batch['extra_features'].to(device)
            overall = batch['overall'].to(device)
            helpful = batch['helpful'].to(device)

            optimizer.zero_grad()
            overall_pred, helpful_pred = model(review_ids, review_mask, summary_ids, summary_mask, extra)
            loss_overall = criterion(overall_pred, overall.view(-1, 1))
            loss_helpful = criterion(helpful_pred, helpful)
            loss = loss_overall + loss_helpful
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, criterion)
        print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1} (best epoch: {best_epoch}, best val loss: {best_val_loss:.4f})')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'Restored best weights from epoch {best_epoch} (val loss: {best_val_loss:.4f})')

In [61]:
train(model, train_loader, val_loader, criterion, optimizer, scheduler=scheduler, epochs=20, patience=4)

Epoch 1, Train Loss: 1.7431, Val Loss: 1.0760
Epoch 2, Train Loss: 1.2187, Val Loss: 1.0442
Epoch 3, Train Loss: 1.1294, Val Loss: 0.9796
Epoch 4, Train Loss: 1.0403, Val Loss: 0.9899
Epoch 5, Train Loss: 0.9729, Val Loss: 0.9879
Epoch 6, Train Loss: 0.9158, Val Loss: 0.9696
Epoch 7, Train Loss: 0.8737, Val Loss: 0.9481
Epoch 8, Train Loss: 0.8296, Val Loss: 0.9958
Epoch 9, Train Loss: 0.7790, Val Loss: 0.9837
Epoch 10, Train Loss: 0.7491, Val Loss: 0.9863
Epoch 11, Train Loss: 0.6711, Val Loss: 0.9769
Early stopping at epoch 11 (best epoch: 7, best val loss: 0.9481)
Restored best weights from epoch 7 (val loss: 0.9481)


## 2. test.jsonl을 입력으로 세 가지 값 예측

In [ ]:
test_df['reviewText'] = test_df['reviewText'].fillna('')
test_df['summary'] = test_df['summary'].fillna('')

test_df['reviewText_token'], test_df['reviewText_mask'] = zip(*test_df['reviewText'].apply(lambda x: preprocessing(x, MAX_LEN_REVIEW)))
test_df['summary_token'], test_df['summary_mask'] = zip(*test_df['summary'].apply(lambda x: preprocessing(x, MAX_LEN_SUMMARY)))

# train_split 통계로 동일하게 보조 feature 생성 (학습 때와 동일한 정규화 기준 사용)
test_df = add_text_features(test_df)
test_df['extra_features'] = list(normalize_features(test_df))

test_df[['row_id', 'reviewText_token', 'reviewText_mask', 'summary_token', 'summary_mask']].head()

In [ ]:
class TestReviewDataset(Dataset):
    def __init__(self, df):
        self.data = df.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        return {
            'row_id': int(row['row_id']),
            'review_ids': torch.tensor(row['reviewText_token'], dtype=torch.long),
            'review_mask': torch.tensor(row['reviewText_mask'], dtype=torch.long),
            'summary_ids': torch.tensor(row['summary_token'], dtype=torch.long),
            'summary_mask': torch.tensor(row['summary_mask'], dtype=torch.long),
            'extra_features': torch.tensor(row['extra_features'], dtype=torch.float)
        }

test_dataset = TestReviewDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
model.eval()
pred_map = {}

with torch.no_grad():
    for batch in test_loader:
        row_ids = batch['row_id'].tolist()
        review_ids = batch['review_ids'].to(device)
        review_mask = batch['review_mask'].to(device)
        summary_ids = batch['summary_ids'].to(device)
        summary_mask = batch['summary_mask'].to(device)
        extra = batch['extra_features'].to(device)

        overall_pred, helpful_pred = model(review_ids, review_mask, summary_ids, summary_mask, extra)
        overall_pred = overall_pred.squeeze(-1).cpu().numpy()
        # helpful은 log1p 스케일로 학습했으므로 expm1로 원래 표 수 스케일로 복원
        helpful_pred = np.expm1(helpful_pred.cpu().numpy().clip(-5, 10))

        for rid, o, hp in zip(row_ids, overall_pred, helpful_pred):
            pred_map[rid] = {
                'helpful_yes_pred': float(hp[0]),
                'helpful_total_pred': float(hp[1]),
                'overall_pred': float(o)
            }

len(pred_map)

## 3. submission.csv에 예측 결과 저장

In [86]:
submission_df['helpful_yes_pred'] = submission_df['row_id'].map(lambda r: pred_map.get(int(r), {}).get('helpful_yes_pred'))
submission_df['helpful_total_pred'] = submission_df['row_id'].map(lambda r: pred_map.get(int(r), {}).get('helpful_total_pred'))
submission_df['overall_pred'] = submission_df['row_id'].map(lambda r: pred_map.get(int(r), {}).get('overall_pred'))

missing = submission_df[submission_df[['helpful_yes_pred', 'helpful_total_pred', 'overall_pred']].isna().any(axis=1)]

# 평가 지표 왜곡 방지를 위해 값의 정의역에 맞게 클리핑
# (overall: 1~5점, helpful_total/yes: 음수 불가, yes는 total을 넘을 수 없음)
submission_df['overall_pred'] = submission_df['overall_pred'].clip(1, 5)
submission_df['helpful_total_pred'] = submission_df['helpful_total_pred'].clip(lower=0)
submission_df['helpful_yes_pred'] = submission_df[['helpful_yes_pred', 'helpful_total_pred']].min(axis=1).clip(lower=0)

submission_df.head()

# Drive 경로 저장이 불안정할 수 있어, 로컬에 저장한 뒤 브라우저로 바로 다운로드
output_path = 'submission.csv'
submission_df.to_csv(output_path, index=False)

from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>